# Класифікація спаму: Naive Bayes

**Мультиноміальна модель подій**

Ми застосуємо наївний Баєсівський класифікатор зі згладжуванням Лапласа для навчання спам-фільтру на основі даних [SpamAssassin Public Corpus](http://spamassassin.apache.org/publiccorpus/).

Заповніть пропущений код (позначено коментарями) та визначте точність передбачення. У вас повинен вийти кращий результат, ніж при моделі багатовимірного розподілу Бернуллі.

In [7]:
import json

In [8]:
import numpy as np

from sklearn.model_selection import train_test_split
from tqdm import tqdm_notebook as progressbar

## Завантаження даних

Щоб краще зрозуміти, яким чином були очищені дані, див. [`spam-data-preparation.ipynb`](spam-data-preparation.ipynb).

In [9]:
def load_json_from_file(filename):
    with open(filename, "r", encoding="utf-8") as f:
        return json.load(f)

In [10]:
emails_tokenized_ham = load_json_from_file("emails-tokenized-ham.json")
emails_tokenized_spam = load_json_from_file("emails-tokenized-spam.json")

In [11]:
vocab = load_json_from_file("vocab.json")

## Кодування даних

Представте кожен лист як $n$-вимірний вектор $\left[ x_1, x_2, ..., x_n \right]$, де $x_i$ — це індекс $i$-го слова даного листа у словнику $V$, а $n$ — кількість слів у листі.

Наприклад, лист _"Buy gold watches. Buy now."_ міг би бути закодований так: $\left[ 3953, 11890, 32213, 3953, 20330 \right]$.

In [12]:
def email_to_vector_multinomial(email_words, vocab):
    # =============== TODO: Your code here ===============
    # Build a feature vector for a single email using the
    # multinomial event model.
    indices = []
    for word in email_words:
        if word in vocab:
            indices.append(vocab[word])
    return np.array(indices, dtype=np.int64)
    # ====================================================

Тепер закодуємо всі листи:

In [13]:
X = [
    email_to_vector_multinomial(email, vocab)
    for email in emails_tokenized_ham + emails_tokenized_spam
]

In [14]:
y = np.array([0] * len(emails_tokenized_ham) + [1] * len(emails_tokenized_spam))

Поглянемо на кілька випадкових листів:

In [15]:
sample_emails = [emails_tokenized_ham[10], emails_tokenized_ham[70]]

In [16]:
for email in sample_emails:
    print(email)
    print()

['hello', 'seen', 'discuss', 'articl', 'approach', 'thank', 'httpaddress', 'hell', 'rule', 'tri', 'accomplish', 'someth', 'thoma', 'alva', 'edison', 'sf', 'net', 'email', 'sponsor', 'osdn', 'tire', 'old', 'cell', 'phone', 'get', 'new', 'free', 'httpaddress', 'spamassassin', 'devel', 'mail', 'list', 'emailaddress', 'httpaddress']

['fri', 'number', 'aug', 'number', 'tom', 'wrote', 'xvid', 'number', 'project', 'make', 'gpl', 'divx', 'codec', 'sigma', 'design', 'number', 'sorri', 'sigma', 'design', 'number', 'number', 'httpaddress']



In [17]:
for email in sample_emails:
    email_vec = email_to_vector_multinomial(email, vocab)
    
    print("Email vector:", email_vec)
    print("Dimensionality:", email_vec.shape)
    print()

Email vector: [12866 26186  7632  1574  1361 29410 13468 12862 25408 30214   173 27396
 29564   872  8567 26411 19758  8849 27722 21116 29770 20748  4549 22215
 11528 19855 10871 13468 27540  7314 17535 16947  8851 13468]
Dimensionality: (34,)

Email vector: [10946 20419  1845 20419 29891 33008 33256 20419 23298 17594 12021  7779
  5370 26756  7232 20419 27443 26756  7232 20419 20419 13468]
Dimensionality: (22,)



## Розділення вибірок

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

In [19]:
print("# Train:", len(X_train))
print("# Test: ", len(X_test))

# Train: 4987
# Test:  555


## Навчання наївного Баєсового класифікатора

Підрахуйте сумарну кількість слів у ham- і spam-листах відповідно.

In [32]:
# =============== TODO: Your code here ===============
# Count the total number of words in ham and spam emails.
# Store these counts into the two variables defined below.

ham_total_words_train = 0
spam_total_words_train = 0

for i in range(len(X_train)):
    if y_train[i] == 0:
        ham_total_words_train += len(X_train[i])
    else:
        spam_total_words_train += len(X_train[i])

print(ham_total_words_train)
print(spam_total_words_train)
# ====================================================

796332
573036


Тепер обчисліть апріорні імовірності для класів ham і spam. Зауважте, що добуток імовірностей може переповнити тип даних змінної, тому ми будемо використовувати логарифми.

In [33]:
# =============== TODO: Your code here ===============
# Compute the class priors for ham and spam emails.

m_train = len(y_train)

ham_count = np.sum(y_train == 0)
spam_count = np.sum(y_train == 1)

ham_log_prior = np.log(ham_count / m_train)
spam_log_prior = np.log(spam_count / m_train)
print(ham_log_prior)
print(spam_log_prior)
# ====================================================

-0.3407323507725008
-1.2421914129760763


Обчисліть правдоподібності (likelihood) для кожного слова. Також, застосуйте згладжування Лапласа, щоб уникнути ділення на нуль.

Створимо порожні вектори $\log{\phi_{word \, | \, ham}}$ та $\log{\phi_{word \, | \, spam}}$ і заповнимо їх для кожного слова зі словника.

In [36]:
ham_log_phi = np.zeros(len(vocab), dtype="float64")
spam_log_phi = np.zeros(len(vocab), dtype="float64")

In [37]:
ham_word_counts = np.zeros(len(vocab))
spam_word_counts = np.zeros(len(vocab))

In [40]:
# =============== TODO: Your code here ===============
# Compute log phi(word | class) for each word in the vocabulary.
# Fill out the `ham_log_phi` and `spam_log_phi` arrays below.

V = len(vocab)

ham_word_counts = np.zeros(V, dtype=np.float64)
spam_word_counts = np.zeros(V, dtype=np.float64)

for i in range(len(X_train)):
    email_vec = X_train[i]
    if len(email_vec) == 0:
        continue

    counts = np.bincount(email_vec, minlength=V).astype(np.float64)

    if y_train[i] == 0:
        ham_word_counts += counts
    else:
        spam_word_counts += counts

ham_log_phi = np.log((ham_word_counts + 1.0) / (ham_total_words_train + V))
spam_log_phi = np.log((spam_word_counts + 1.0) / (spam_total_words_train + V))

print(ham_log_phi.shape)
print(spam_log_phi.shape)
# ====================================================

(34133,)
(34133,)


## Передбачення

Реалізуйте функцію передбачення. Пригадайте, що знаменник $P(words)$ — один і той самий для обох класів, тому для передбачення його можна проігнорувати.

In [41]:
def predict(X):
    # =============== TODO: Your code here ===============
    # Implement the prediction of target classes, given
    # a feature dataset X. You should return a response
    # vector containing n {0, 1} values, where n is the
    # number of examples in X.
    
    preds = np.zeros(len(X), dtype=np.int64)

    for i in range(len(X)):
        email_vec = X[i]

        ham_score = ham_log_prior
        spam_score = spam_log_prior

        if len(email_vec) > 0:
            ham_score += np.sum(ham_log_phi[email_vec])
            spam_score += np.sum(spam_log_phi[email_vec])

        preds[i] = 1 if spam_score > ham_score else 0

    return preds
    # ====================================================

In [42]:
print("ham_total_words_train:", ham_total_words_train)
print("spam_total_words_train:", spam_total_words_train)

print("ham_word_counts sum:", ham_word_counts.sum())
print("spam_word_counts sum:", spam_word_counts.sum())

print("ham nonzero:", np.count_nonzero(ham_word_counts))
print("spam nonzero:", np.count_nonzero(spam_word_counts))

ham_total_words_train: 796332
spam_total_words_train: 573036
ham_word_counts sum: 796332.0
spam_word_counts sum: 573036.0
ham nonzero: 25893
spam nonzero: 13904


## Оцінка точності передбачення

In [43]:
pred_train = predict(X_train)
pred_test = predict(X_test)

In [44]:
accuracy_train = 1 - np.sum(pred_train != y_train) / len(y_train)
accuracy_test = 1 - np.sum(pred_test != y_test) / len(y_test)

In [45]:
print("Training accuracy:   {0:.3f}%".format(accuracy_train * 100))
print("Test accuracy:       {0:.3f}%".format(accuracy_test * 100))

Training accuracy:   97.654%
Test accuracy:       97.477%
